# Hybrid Search with Dense and Sparse Vectors in Milvus

如果你想体验本教程的最终效果，可以直接访问 https://demos.milvus.io/hybrid-search/

在本教程中，我们将演示如何使用 Milvus 与 BGE-M3 模型进行混合搜索。BGE-M3 模型能够将文本转换为密集向量和稀疏向量。Milvus 支持将这两种类型的向量存储在同一集合中，从而实现混合搜索，提升搜索结果的相关性。

Milvus 支持密集检索、稀疏检索和混合检索三种方式：

- 密集检索：利用语义上下文理解查询背后的含义。
- 稀疏检索：侧重关键词匹配，根据特定术语查找结果，相当于全文搜索。
- 混合检索：结合密集检索和稀疏检索方法，兼顾整体语义和具体关键词，实现全面的搜索结果。

通过整合这些检索策略，Milvus 混合搜索能够在语义相似性和词汇相似性之间取得平衡，从而提高搜索结果的整体相关性。本笔记将逐步介绍如何设置并使用这些检索策略，并重点展示其在不同搜索场景中的有效性。

## Dependencies and Environment


In [1]:
# !pip install --upgrade pymilvus "pymilvus[model]"

In [2]:
import os

os.environ['NLTK_DATA'] = r'F:\Teewon\Milvue\model\nltk_data'

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [3]:
import dotenv
dotenv.load_dotenv('../.env')

True

## 下载数据集
为了演示搜索功能，我们需要一个文档语料库。我们使用 Quora 双重问题数据集，并将其放置到本地目录中。

数据集来源：[First Quora Dataset Release: Question Pairs](https://www.quora.com/q/quoradata/First-Quora-Dataset-Release-Question-Pairs)

## 加载并准备数据
我们将加载数据集，并为搜索准备一个小型语料库。

In [4]:
import pandas as pd

file_path = "./data/train.csv"
df = pd.read_csv(file_path)
# 删除 question1 或 question2 为 NaN 的行
df_clean = df.dropna(subset=['question1', 'question2'])

questions = set()
for _, row in df_clean.iterrows():
    questions.add(row["question1"][:512])
    questions.add(row["question2"][:512])
    # 如果需要限制数量，可以取消注释
    # if len(questions) > 500:
    #     break

docs = list(questions)
print(docs[0])
#使用 questions = set() 来存储问题，集合（set）本身不保证任何顺序。当你将集合转换为列表 docs = list(questions) 时，元素的排列顺序是任意的（基于哈希值），与数据读取顺序无关。

Mrs. Clinton: What are your goals/plans for NASA and the space program?


## Use BGE-M3 Model for Embeddings
The BGE-M3 model can embed texts as dense and sparse vectors.

In [5]:
from pymilvus.model.hybrid import BGEM3EmbeddingFunction

ef = BGEM3EmbeddingFunction(use_fp16=False, device="cuda")
dense_dim = ef.dim["dense"]

# Generate embeddings using BGE-M3 model
docs_embeddings = ef(docs)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 33585/33585 [00:15<00:00, 2155.86it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 33585/33585 [21:46<00:00, 25.70it/s]


## 设置 Milvus 集合和索引
我们将创建 Milvus 集合，并为向量字段建立索引。

> - 将 URI 设置为本地文件（例如 "./milvus.db"）是最方便的方法，因为这会自动使用 Milvus Lite 将所有数据存储在该文件中。
> - 如果您的数据量较大，例如超过一百万个向量，建议在 Docker 或 Kubernetes 上部署性能更高的 Milvus 服务器。在这种情况下，请使用服务器的 URI（例如 http://localhost:19530）作为您的 URI。
> - 如果您想使用 Zilliz Cloud（Milvus 的全托管云服务），请相应调整 URI 和令牌，这些对应于 Zilliz Cloud 中的公共端点和 API 密钥。

In [7]:
from pymilvus import (
    MilvusClient,
    DataType,
    Function,
    FunctionType,
    AnnSearchRequest,
    RRFRanker
)
milvus_uri="http://localhost:19530"
mc=MilvusClient(milvus_uri)
collection_name='hybrid_search_demo'
mc=MilvusClient(milvus_uri,collection_name)
print(mc.get_server_version())

3.0.0


In [8]:
analyzer_params={"tokenizer":"standard","filter":["lowercase"]}

In [13]:
schema=MilvusClient.create_schema()
schema.add_field(
    field_name='pk',
    datatype=DataType.VARCHAR,
    is_primary=True,
    auto_id=True,
    max_length=100,
)
schema.add_field(
    field_name='text',
    datatype=DataType.VARCHAR,
    max_length=512,
    analyzer_params=analyzer_params,
    enable_match=True,  # Enable text matching
    enable_analyzer=True,     # Enable text analysis
)
schema.add_field(
    field_name='sparse_vector',
    datatype=DataType.SPARSE_FLOAT_VECTOR,
)
schema.add_field(
    field_name='dense_vector',
    datatype=DataType.FLOAT_VECTOR,
    dim=dense_dim   # Dimension for text-embedding-3-small
)

index_params=MilvusClient.prepare_index_params()
index_params.add_index(
    field_name='sparse_vector',
    index_type="SPARSE_INVERTED_INDEX",
    metric_type= "IP"
)
index_params.add_index(
    field_name='dense_vector',
    index_type="FLAT",
    metric_type= "IP"
)

In [14]:
if mc.has_collection(collection_name):
    mc.drop_collection(collection_name)

mc.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params,
)

In [39]:
from scipy.sparse import issparse, csr_matrix
import numpy as np

def ensure_single_row_sparse(vec):
    """将任何稀疏向量转换为 (1, N) 的 csr_matrix"""
    if issparse(vec):
        # 先确保是二维（如果是 1D 则 reshape）
        if vec.ndim == 1:
            vec = vec.reshape(1, -1)
        # 强制转换为 csr_matrix（确保有 .indices）
        return csr_matrix(vec)
    elif isinstance(vec, np.ndarray):
        if vec.ndim == 1:
            return csr_matrix(vec.reshape(1, -1))
        elif vec.ndim == 2 and vec.shape[0] != 1:
            return csr_matrix(vec.reshape(1, -1))
        return csr_matrix(vec)
    # 其他（如字典）原样返回
    return vec

In [33]:
batch_size = 500
for i in range(0,len(docs),batch_size):
    batch_texts=docs[i:i+batch_size]
    batch_sparse = docs_embeddings["sparse"][i : i + batch_size]
    batch_dense = docs_embeddings['dense'][i:i+batch_size]

    entities=[]
    for j in range(len(batch_texts)):
        sparse_vec = ensure_single_row_sparse(batch_sparse[j])
        entities.append({
            "text": batch_texts[j],
            "sparse_vector": sparse_vec,
            "dense_vector": batch_dense[j],
        })
    mc.insert(collection_name, entities)

stats=mc.get_collection_stats(collection_name)
entity_count=stats['row_count']
print("Number of entities inserted: ",entity_count)

Number of entities inserted:  534500


## Enter Your Search Query

In [34]:
# Enter your search query
query = input("Enter your search query: ")
print(query)

# Generate embeddings for the query
query_embeddings = ef([query])
# print(query_embeddings)

How to start learning programming?


## 运行搜索
我们将首先准备一些实用函数来执行搜索：

- `dense_search`：仅在密集向量场中进行搜索
- `sparse_search`：仅在稀疏向量场中进行搜索
- `hybrid_search`：结合密集和向量场进行搜索，并使用加权重排序

In [42]:
from pymilvus import AnnSearchRequest,WeightedRanker

def dense_search(mc,query_dense_embeddings,limit=10):
    search_params={"metric_type":"IP","params":{}}
    res=mc.search(
        collection_name=collection_name,
        data=[query_dense_embeddings],
        anns_field="dense_vector",
        search_params=search_params,
        limit=limit,
        output_fields=["text"],
    )[0]
    return [hit['entity']['text'] for hit in res]

def sparse_search(mc,query_sparse_embeddings,limit=10):
    query_vector=ensure_single_row_sparse(query_sparse_embeddings)
    search_params={"metric_type":"IP","params":{}}
    res=mc.search(
        collection_name=collection_name,
        data=[query_vector],
        anns_field="sparse_vector",
        search_params=search_params,
        limit=limit,
        output_fields=["text"],
    )[0]
    return [hit['entity']['text'] for hit in res]

def hybrid_search(mc,query_dense_embeddings,query_sparse_embeddings,sparse_weight,dense_weight,limit=10):
    dense_search_params={"metric_type":"IP","params":{}}
    dense_req=AnnSearchRequest(
        [query_dense_embeddings],
        "dense_vector",
        dense_search_params,
        limit=limit,
    )
    sparse_search_params={"metric_type":"IP","params":{}}
    sqarse_vector=ensure_single_row_sparse(query_sparse_embeddings)
    sparse_req=AnnSearchRequest(
        [sqarse_vector],
        "sparse_vector",
        sparse_search_params,
        limit=limit,
    )
    rerank=WeightedRanker(sparse_weight,dense_weight)
    res=mc.hybrid_search(
        collection_name,
        [sparse_req,dense_req],
        ranker=rerank,
        limit=limit,
        output_fields=["text"],
    )[0]
    return [hit['entity']['text'] for hit in res]

In [43]:
dense_results=dense_search(mc,query_embeddings["dense"][0])
sparse_results=sparse_search(mc,query_embeddings["sparse"][0])
hybrid_results=hybrid_search(mc,query_embeddings["dense"][0],query_embeddings["sparse"][0],sparse_weight=0.7,dense_weight=1.0)

## 显示搜索结果
为了显示密集、稀疏和混合搜索的结果，我们需要一些工具来格式化这些结果。

In [45]:
def doc_text_formatting(ef,query,docs):
    tokenizer=ef.model.tokenizer
    query_tokens_ids=tokenizer.encode(query,return_offsets_mapping=True)
    query_tokens=tokenizer.convert_ids_to_tokens(query_tokens_ids)
    formatted_texts=[]

    for doc in docs:
        ldx=0
        landmarks=[]
        encoding=tokenizer.encode_plus(doc,return_offsets_mapping=True)
        tokens=tokenizer.convert_ids_to_tokens(encoding["input_ids"])[1:-1]
        offsets=encoding["offset_mapping"][1:-1]
        for token,(start,end) in zip(tokens,offsets):
            if token in query_tokens:
                if len(landmarks)!=0 and start==landmarks[-1]:
                    landmarks[-1]=end
                else:
                    landmarks.append(start)
                    landmarks.append(end)
        close=False
        formatted_text=""
        for i,c in enumerate(doc):
            if ldx==len(landmarks):
                pass
            elif i==landmarks[ldx]:
                if close:
                    formatted_text+="</span>"
                else:
                    formatted_text+="<span style='color:red'>"
                close=not close
                ldx=ldx+1
            formatted_text+=c
        if close is True:
            formatted_text+="</span>"
        formatted_texts.append(formatted_text)
    return formatted_texts

Then we can display search results in text with highlights:

In [47]:
from IPython.display import Markdown,display

display(Markdown("**Dense Search Results**"))
formatted_results=doc_text_formatting(ef,query,dense_results)
for result in formatted_results:
    display(Markdown(result))

# Sparse search results
display(Markdown("\n**Sparse Search Results:**"))
formatted_results = doc_text_formatting(ef, query, sparse_results)
for result in formatted_results:
    display(Markdown(result))

# Hybrid search results
display(Markdown("\n**Hybrid Search Results:**"))
formatted_results = doc_text_formatting(ef, query, hybrid_results)
for result in formatted_results:
    display(Markdown(result))

**Dense Search Results**

<span style='color:red'>How</span> do I<span style='color:red'> start learning programming?</span>

<span style='color:red'>How</span> should you<span style='color:red'> start learning programming?</span>

<span style='color:red'>How</span> should I begin<span style='color:red'> to</span> learn<span style='color:red'> programming?</span>

<span style='color:red'>How</span> do you get started<span style='color:red'> learning programming?</span>

...

From where do I<span style='color:red'> start learning programming?</span>

<span style='color:red'>How</span> should I begin<span style='color:red'> to</span> learn<span style='color:red'> programming?</span>

<span style='color:red'>How</span> do you get started<span style='color:red'> learning programming?</span>

<span style='color:red'>How</span> should I<span style='color:red'> start</span> if I want<span style='color:red'> to</span> learn<span style='color:red'> programming?</span>

<span style='color:red'>How</span> should I get started with<span style='color:red'> programming?</span>

In [48]:
mc.drop_collection(collection_name)